# Exception Handling and Recovery

Agents deployed in real-world environments encounter tool failures, network errors, and invalid responses. The Exception Handling pattern provides layered resilience: detect errors → retry transient failures → fall back to an alternative strategy → degrade gracefully → escalate if all else fails.

## Implementation with Flyte v2

This notebook reimplements the ADK `SequentialAgent` (primary → fallback → response) pattern from Chapter 12 using **Flyte v2 primitives**.

#### ADK vs Flyte v2 — Key Differences

| Aspect | ADK | Flyte v2 |
|--------|-----|----------|
| **Retry logic** | None built-in at agent level | `retries=N` on `@env.task` — automatic exponential backoff |
| **Fallback routing** | `SequentialAgent` reads `state["primary_*_failed"]` | Plain Python `try/except` with typed fallback path |
| **Error state** | Mutable shared `state` dict | Immutable `RecoveryResult` dataclass — serializable, inspectable |
| **Graceful degradation** | Separate `response_agent` checks state | Inline conditional: return partial result or error message |
| **Escalation** | Not standardized | `RecoveryResult(escalate=True)` + outer loop checks flag |
| **Observability** | State dict inspection | Typed fields in Flyte UI (`primary_succeeded`, `fallback_used`, etc.) |
| **Secrets** | `os.environ` at import time | `flyte.Secret` injected at task runtime |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from datetime import timedelta
from typing import Optional

from anthropic import AsyncAnthropic
import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="recovery-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0")
)

recovery_env = flyte.TaskEnvironment(
    name="recovery_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define data models

The ADK example passes error state through a mutable `state` dict shared across agents. In Flyte v2, `RecoveryResult` is an immutable typed dataclass — each field explicitly captures which recovery path was taken, making the decision fully inspectable in the UI without parsing logs.

In [ ]:
@dataclass
class LocationInfo:
    """Location data — may be precise or general depending on which tool succeeded."""
    query: str
    latitude: Optional[float]
    longitude: Optional[float]
    city: Optional[str]
    precision: str   # "precise" | "general" | "unavailable"


@dataclass
class RecoveryResult:
    """Outcome of the exception-handling agent run."""
    query: str
    location: Optional[LocationInfo]
    primary_succeeded: bool
    fallback_used: bool
    error_log: list[str]
    escalate: bool
    final_response: str

### 5. Define primary and fallback tools

The ADK example has `get_precise_location_info` (primary) and `get_general_area_info` (fallback) as tool functions, with the `primary_handler` agent trying the first and the `fallback_handler` checking `state["primary_location_failed"]`. In Flyte v2, this is explicit `try/except` — same semantics, fully transparent.

We simulate intermittent failures to demonstrate recovery. `retries=2` on the primary task handles transient errors automatically before the fallback is triggered.

In [ ]:
class PreciseLocationError(Exception):
    """Simulates an external geocoding API failure."""


def _get_precise_location(address: str) -> LocationInfo:
    """
    Simulate calling a precise geocoding API that sometimes fails.
    In production: replace with an actual geocoding API call.
    """
    # Simulate ~30% failure rate to demonstrate fallback
    if random.random() < 0.3 or "unknown" in address.lower():
        raise PreciseLocationError(f"Geocoding API returned 503 for: {address!r}")

    # Simulate success — return plausible coordinates
    return LocationInfo(
        query=address,
        latitude=37.7749,
        longitude=-122.4194,
        city=None,
        precision="precise",
    )


def _get_general_area(address: str) -> LocationInfo:
    """
    Fallback: extract city from address string without calling an external API.
    More resilient but less precise — equivalent to get_general_area_info in ADK.
    """
    parts = [p.strip() for p in address.split(",") if p.strip()]
    city = parts[-2] if len(parts) >= 2 else parts[0] if parts else address
    return LocationInfo(
        query=address,
        latitude=None,
        longitude=None,
        city=city,
        precision="general",
    )


async def _compose_response(location: LocationInfo, client: AsyncAnthropic) -> str:
    """Use the LLM to compose a user-friendly response from location data."""
    if location.precision == "precise":
        context = f"Precise coordinates: {location.latitude:.4f}, {location.longitude:.4f}"
    elif location.precision == "general":
        context = f"General area: {location.city}"
    else:
        context = "No location data could be retrieved."

    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=128,
        messages=[{
            "role": "user",
            "content": (
                f"User asked about: {location.query!r}\n"
                f"Location result: {context}\n\n"
                "Write a helpful one-sentence response to the user."
            ),
        }],
    )
    return response.content[0].text.strip()

### 6. Define the exception-handling agent task

The ADK `SequentialAgent` runs three agents in order — `primary_handler → fallback_handler → response_agent` — with shared mutable state. In Flyte v2, the same pipeline is one task with explicit `try/except` control flow:

1. **Primary path** (with retries): call precise geocoding API
2. **Fallback path**: if primary fails after retries, extract city from address string
3. **Graceful degradation**: if both fail, return an `escalate=True` result
4. **Response**: LLM composes a user-friendly message from whatever data is available

Flyte's `retries=2` handles transient failures before the fallback is triggered — equivalent to the `primary_handler` retrying before the `fallback_handler` kicks in.

In [ ]:
@recovery_env.task(
    retries=2,
    timeout=timedelta(minutes=2),
    cache=flyte.Cache(behavior="disable"),
)
async def location_agent(address: str) -> RecoveryResult:
    """
    Robust location retrieval with primary → fallback → escalation.

    Replaces ADK's:
      SequentialAgent(sub_agents=[primary_handler, fallback_handler, response_agent])

    Key difference: instead of routing through shared mutable state, each recovery
    decision is an explicit code path — easier to reason about and test.
    """
    client = AsyncAnthropic()
    error_log: list[str] = []
    location: Optional[LocationInfo] = None
    primary_succeeded = False
    fallback_used = False

    # ── Primary path ──────────────────────────────────────────────────────────
    # Equivalent to primary_handler agent calling get_precise_location_info
    try:
        location = _get_precise_location(address)
        primary_succeeded = True
    except PreciseLocationError as e:
        error_log.append(f"Primary failed: {e}")

        # ── Fallback path ─────────────────────────────────────────────────────
        # Equivalent to fallback_handler agent checking state["primary_location_failed"]
        try:
            location = _get_general_area(address)
            fallback_used = True
            error_log.append("Fallback: using general area extraction")
        except Exception as fe:
            error_log.append(f"Fallback also failed: {fe}")

    # ── Graceful degradation / escalation ────────────────────────────────────
    if location is None:
        location = LocationInfo(
            query=address, latitude=None, longitude=None,
            city=None, precision="unavailable",
        )
        return RecoveryResult(
            query=address,
            location=location,
            primary_succeeded=False,
            fallback_used=False,
            error_log=error_log,
            escalate=True,  # signal: needs human review
            final_response="Unable to retrieve location. Escalated for manual review.",
        )

    # ── Response composition ──────────────────────────────────────────────────
    # Equivalent to response_agent reviewing state["location_result"]
    final_response = await _compose_response(location, client)

    return RecoveryResult(
        query=address,
        location=location,
        primary_succeeded=primary_succeeded,
        fallback_used=fallback_used,
        error_log=error_log,
        escalate=False,
        final_response=final_response,
    )

### 7. Run locally — demonstrate recovery paths

In [ ]:
TEST_ADDRESSES = [
    "1600 Amphitheatre Parkway, Mountain View, CA",      # likely primary succeeds
    "unknown location, no zip",                          # triggers fallback
    "350 5th Avenue, New York, NY 10118",                # likely primary succeeds
    "somewhere, USA",                                    # general area fallback
]

for address in TEST_ADDRESSES:
    run = flyte.run(location_agent, address=address)
    run.wait()
    result: RecoveryResult = run.outputs()[0]

    path = "primary" if result.primary_succeeded else ("fallback" if result.fallback_used else "escalated")
    precision = result.location.precision if result.location else "none"
    print(f"Address: {address[:50]}")
    print(f"  Path: {path} | Precision: {precision} | Escalate: {result.escalate}")
    print(f"  Response: {result.final_response}")
    if result.error_log:
        print(f"  Errors: {result.error_log}")
    print()

### Running remotely

The `RecoveryResult` dataclass appears as structured output in the Flyte UI — `primary_succeeded`, `fallback_used`, `escalate`, and `error_log` are all inspectable fields, making the recovery decision auditable without log parsing. Tasks with `retries=2` automatically retry transient failures with exponential backoff before triggering the fallback path.

In [ ]:
run = flyte.run(location_agent, address="One Microsoft Way, Redmond, WA 98052")
run.wait()
result = run.outputs()[0]
print(f"Primary succeeded: {result.primary_succeeded}")
print(f"Fallback used: {result.fallback_used}")
print(f"Precision: {result.location.precision if result.location else 'none'}")
print(result.final_response)

## Why not the Agent harness here?

Exception handling maps onto Flyte's own machinery: `retries=` on `@env.task`, `try/except` with `.override(resources=...)`, and the typed exceptions in `flyte.errors` (e.g. `OOMError`). These are deterministic control-flow concerns, not decisions for an LLM. An `Agent` wouldn't make failures any more recoverable — it would only blur where the retry boundary lives.